# Pielasa Direct Sum for Complete Graphs

Implements the Hilbert series formula from [Pielasa, arXiv:2511.10746]:

$$H_{K_a \oplus K_b \oplus \cdots}(x) = H_M(x) H_N(x) + x \sum_{F,G} H_{M_F \oplus N_G}(x) \cdot H_{M^F}(x) \cdot H_{N^G}(x)$$

where the sum is over pairs of proper flats F, G with $F < M$ and $G< N$.

For $K_n$, flats correspond to partitions of $[n]$: flat with block sizes $\lambda$ gives $M_F = K_{\lambda_1} \oplus \cdots \oplus K_{\lambda_r}$, $M^F = K_r$.

In [37]:
import sys
import os
import itertools
import json
from collections import Counter

current_dir = os.path.abspath(os.getcwd())
if "explorations on complete graphs" in current_dir:
    project_root = os.path.dirname(os.path.dirname(current_dir))
else:
    project_root = current_dir
sys.path.insert(0, project_root)
sys.path.insert(0, os.path.join(project_root, "the-machine"))

from sage.all import *
from custom_matroid_functions import get_chow_polynomial

# Polynomial ring and chowBraid path
P = LaurentPolynomialRing(QQ, "x")
x = P.gen()
_expl_dir = current_dir if "explorations on complete graphs" in current_dir else os.path.join(project_root, "conclusion", "explorations on complete graphs")
CHOWBRAID_PATH = os.path.join(_expl_dir, "chowBraid.json")

# Caches (loaded from chowBraid.json)
H_Kn = {}
H_direct_sum = {}

def _parse_poly(s):
    if s == "1" or s == 1:
        return P(1)
    return P(str(s))

def _load_chowBraid():
    global H_Kn, H_direct_sum
    if not os.path.exists(CHOWBRAID_PATH):
        return
    try:
        with open(CHOWBRAID_PATH, "r") as f:
            data = json.load(f)
        H_Kn.clear()
        for k, v in data.get("H_Kn", {}).items():
            H_Kn[int(k)] = _parse_poly(v)
        H_direct_sum.clear()
        for k, v in data.get("H_direct_sum", {}).items():
            H_direct_sum[tuple(json.loads(k))] = _parse_poly(v)
    except (json.JSONDecodeError, ValueError):
        pass

def _save_chowBraid():
    try:
        h_kn = {str(int(k)): str(v) for k, v in H_Kn.items()}
        h_ds = {json.dumps([int(x) for x in k]): str(v) for k, v in H_direct_sum.items()}
        with open(CHOWBRAID_PATH, "w") as f:
            json.dump({"H_Kn": h_kn, "H_direct_sum": h_ds}, f, indent=2)
    except Exception:
        pass

_load_chowBraid()

def _get_H_Kn(n, kn):
    """Return H_{K_n}; compute via get_chow_polynomial if not in cache."""
    if n not in kn:
        kn[n] = P(get_chow_polynomial(matroids.CompleteGraphic(n)))
        _save_chowBraid()
    return kn[n]

def C_lambda(lam, n):
    """Number of set partitions of {1..n} with block sizes lam."""
    mult = Counter(lam)
    denom = 1
    for j, mj in mult.items():
        denom *= factorial(int(j)) ** int(mj) * factorial(int(mj))
    return factorial(int(n)) // denom
    
def chow_direct_sum_complete_graphs(shape, H_Kn_oracle=None):
    """
    Computes H(K_{a1} + K_{a2} + ...) using Pielasa Theorem 1.7.
    Sum runs over F < M AND G < N (Strictly Proper).
    """
    kn = H_Kn_oracle if H_Kn_oracle is not None else H_Kn
    
    # 1. Normalize Shape (Filter 1s, Sort)
    shape = tuple(sorted([int(s) for s in shape if s > 1]))
    
    # Base Cases
    if not shape: return P(1)
    if shape in H_direct_sum: return H_direct_sum[shape]
    if len(shape) == 1:
        # Base case logic must ensure H_Kn is loaded/computed
        return _get_H_Kn(shape[0], kn)

    # 2. Split M = K_{a1} (First), N = Rest (Direct Sum)
    a1, rest = shape[0], shape[1:]
    
    # Tensor Product Term: H(M) * H(N)
    # Note: Recursive call for 'rest' handles the N part correctly
    term_base = _get_H_Kn(a1, kn) * chow_direct_sum_complete_graphs(rest, kn)
    
    term_sum = P(0)
    
    # 3. Iterate Strictly Proper Flats F < M (Partition sizes < 2 excluded)
    # Proper means |pi| >= 2 (Since |pi|=1 is the Top flat of K_n)
    for lam in Partitions(a1):
        lam_list = list(lam)
        
        # Check Strict Properness for M
        if len(lam_list) < 2: 
            continue 
            
        c_M = C_lambda(lam_list, a1)
        # Contraction of M is K_{|lam|}. This is a single component, so lookup is fine.
        h_cont_M = _get_H_Kn(len(lam_list), kn)
        
        # 4. Iterate Strictly Proper Flats G < N
        # We generate flats of N via Cartesian product of partitions
        comp_parts = [[list(p) for p in Partitions(r)] for r in rest]
        
        for parts_tuple in itertools.product(*comp_parts):
            
            # Check Strict Properness for N
            # Top of N is when EVERY component is a single block (len 1)
            is_top_N = all(len(p) == 1 for p in parts_tuple)
            
            if is_top_N:
                continue
                
            # Coefficient C_N
            c_N = 1
            for i, p in enumerate(parts_tuple):
                c_N *= C_lambda(p, rest[i])
                
            # --- THE FIX IS HERE ---
            # Construct the shape of the contraction N/G
            # N/G = DirectSum( K_{|p|} for p in parts_tuple )
            contr_shape_N = tuple(len(p) for p in parts_tuple)
            
            # Recursively compute H(N/G) instead of multiplying
            h_cont_N = chow_direct_sum_complete_graphs(contr_shape_N, kn)
            # -----------------------

            # Restriction H(M|F + N|G)
            # Combined shape of all blocks
            combined_restr = list(lam_list) + [b for p in parts_tuple for b in p]
            h_restr = chow_direct_sum_complete_graphs(tuple(combined_restr), kn)
            
            # Add Term
            term_sum += x * (c_M * c_N) * h_restr * (h_cont_M * h_cont_N)
            
    result = term_base + term_sum
    H_direct_sum[shape] = result
    return result

def reload_chowBraid():
    _load_chowBraid()

In [38]:
# Populate chowBraid.json with H_Kn and H_direct_sum for all shapes from partitions of 1..N_MAX
N_MAX = 10
reload_chowBraid()

shapes_to_compute = set()
for n in range(1, N_MAX + 1):
    for lam in Partitions(n):
        shape = tuple(sorted([int(x) for x in lam if x > 1]))
        if shape:
            shapes_to_compute.add(shape)

shapes_sorted = sorted(shapes_to_compute, key=lambda s: (sum(s), len(s), s))
for i, shape in enumerate(shapes_sorted):
    chow_direct_sum_complete_graphs(shape)
    if (i + 1) % 5 == 0:
        _save_chowBraid()
        print(f"Computed {i+1}/{len(shapes_sorted)} shapes...")

_save_chowBraid()
print(f"Done. chowBraid has {len(H_Kn)} H_Kn, {len(H_direct_sum)} H_direct_sum entries.")

Computed 5/41 shapes...
Computed 10/41 shapes...
Computed 15/41 shapes...
Computed 20/41 shapes...
Computed 25/41 shapes...


KeyboardInterrupt: 